Intruksi awal

Buat notebook baru Tugas10_[NPM]_[Nama Lengkap].ipynb, buat SparkSession (jangan lupa spark.conf.set("spark.sql.shuffle.partitions", "4"))

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas_Pertemuan10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


A. Menyiapkan Streaming Source (bobot 15%)

Definisikan skema eksplisit (StructType) untuk data pesanan dengan kolom: pesanan_id (string), nama_cabang (string), nama_menu (string), qty (integer), harga_satuan (integer), waktu_pesan (timestamp). Buat folder simulasi streaming baru bernama tugas10_stream, lalu definisikan readStream dari folder tersebut.

In [17]:
tugas10_stream = "tugas10_stream"
tugas_folder_checkpoint = "tugas_checkpoint_pertemuan10"

shutil.rmtree(tugas10_stream, ignore_errors=True)
shutil.rmtree(tugas_folder_checkpoint, ignore_errors=True)
os.makedirs(tugas10_stream, exist_ok=True)

def_transaksi = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

tugas10_stream = spark.readStream \
    .format("csv") \
    .schema(def_transaksi) \
    .option("header", True) \
    .load(tugas10_stream)

B. Agregasi per Cabang (bobot 20%)

Tambahkan kolom turunan total_harga (qty x harga_satuan) pada df_stream, lalu buat agregasi groupBy("nama_cabang") yang menghitung total total_harga dan jumlah pesanan. Jalankan sebagai streaming query dengan sink memory, queryName="agregasi_cabang".

In [18]:
tambah = tugas10_stream.withColumn(
    "total_harga", 
    col("qty") * col("harga_satuan")
)

agregasi = tambah.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_agregasi = agregasi.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+



C. Agregasi Berbasis Window Waktu (bobot 20%)

Buat streaming query kedua (definisikan ulang readStream dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan. Gunakan sink memory, queryName="agregasi_waktu".

In [19]:
tugas10_stream_2 = spark.readStream \
    .format("csv") \
    .schema(def_transaksi) \
    .option("header", True) \
    .load("tugas10_stream") 

tambah_2 = tugas10_stream_2.withColumn(
    "total_harga", 
    col("qty") * col("harga_satuan")
)

waktu = tambah_2.groupBy(
    window(col("waktu_transaksi"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query_waktu = waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

D. Menguji dengan Data Simulasi (bobot 25%)

Kirim minimal 3 batch data (masing-masing batch berupa satu berkas CSV baru dengan jeda time.sleep() di antaranya) ke folder tugas10_stream, dengan kombinasi nama_cabang yang bervariasi. Setelah setiap batch, tampilkan hasil agregasi_cabang untuk membuktikan angka totalnya terus terakumulasi dengan benar.



In [20]:
import pandas as pd
import time


print("--- Batch 1 ---")
pd.DataFrame({
    "pesanan_id": ["P001", "P002"],
    "nama_cabang": ["Cabang Depok", "Cabang Bandung"],
    "nama_menu": ["Kwetiaw", "Capcai"],
    "qty": [2, 1],
    "harga_satuan": [20000, 15000],
    "waktu_transaksi": ["2026-11-12 10:00:00", "2026-11-12 10:01:00"]
}).to_csv(f"tugas10_stream/batch_1.csv", index=False)

time.sleep(6) 
print("Hasil :")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()



print("--- Batch 2 ---")

pd.DataFrame({
    "pesanan_id": ["P003", "P004"],
    "nama_cabang": ["Cabang Mojokerto", "Cabang Ngawi"],
    "nama_menu": ["Puding coklat", "Ayam Bakar"],
    "qty": [3, 2],
    "harga_satuan": [5000, 25000],
    "waktu_transaksi": ["2026-11-12 10:02:00", "2026-11-12 10:03:00"]
}).to_csv(f"tugas10_stream/batch_2.csv", index=False)

time.sleep(6)
print("Hasil :")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()



print("--- Batch 3 ---")

pd.DataFrame({
    "pesanan_id": ["P005", "P006"],
    "nama_cabang": ["Cabang LA", "Cabang California"],
    "nama_menu": ["Mac and Cheese", "Beef Bacon"],
    "qty": [2, 1],
    "harga_satuan": [12000, 30000],
    "waktu_transaksi": ["2026-11-12 10:04:00", "2026-11-12 10:05:00"]
}).to_csv(f"tugas10_stream/batch_3.csv", index=False)

time.sleep(5)
print("Hasil :")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

--- Batch 1 ---


Hasil :
+--------------+----------------+--------------+
|   nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------+----------------+--------------+
|  Cabang Depok|           40000|             1|
|Cabang Bandung|           15000|             1|
+--------------+----------------+--------------+

--- Batch 2 ---
Hasil :
+----------------+----------------+--------------+
|     nama_cabang|total_pendapatan|jumlah_pesanan|
+----------------+----------------+--------------+
|    Cabang Ngawi|           50000|             1|
|    Cabang Depok|           40000|             1|
|Cabang Mojokerto|           15000|             1|
|  Cabang Bandung|           15000|             1|
+----------------+----------------+--------------+

--- Batch 3 ---
Hasil :
+-----------------+----------------+--------------+
|      nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------------+----------------+--------------+
|     Cabang Ngawi|           50000|             1|
|     Cabang Depok|           40

E. Sink Berbasis Berkas (Tantangan) (bobot 20%)

Selain memory sink, Structured Streaming juga mendukung menulis hasil DataFrame mentah (bukan hasil agregasi) ke berkas secara berkelanjutan. Buat streaming query ketiga yang membaca dari df_stream, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal tugas10_output dalam format Parquet, menggunakan:

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()


Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (!ls tugas10_output) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan spark.read.parquet().

In [23]:
mentah = tugas10_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

spark.sql("SELECT * FROM tabel_mentah ORDER BY waktu_transaksi ASC").show(truncate=False)

+----------+-----------------+--------------+---+------------+-------------------+
|pesanan_id|nama_cabang      |nama_menu     |qty|harga_satuan|waktu_transaksi    |
+----------+-----------------+--------------+---+------------+-------------------+
|P001      |Cabang Depok     |Kwetiaw       |2  |20000       |2026-11-12 10:00:00|
|P002      |Cabang Bandung   |Capcai        |1  |15000       |2026-11-12 10:01:00|
|P003      |Cabang Mojokerto |Puding coklat |3  |5000        |2026-11-12 10:02:00|
|P004      |Cabang Ngawi     |Ayam Bakar    |2  |25000       |2026-11-12 10:03:00|
|P005      |Cabang LA        |Mac and Cheese|2  |12000       |2026-11-12 10:04:00|
|P006      |Cabang California|Beef Bacon    |1  |30000       |2026-11-12 10:05:00|
+----------+-----------------+--------------+---+------------+-------------------+

